In [3]:
import pandas as pd


In [4]:
df = pd.read_csv("online_food_delivery_dataset.csv")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])


Rows: 388
Columns: 14


In [5]:
df.head()


,Age,Gender,Marital Status,Occupation,Monthly Income,Educational Qualifications,Family size,Customer Type,latitude,longitude,Pin code,Output,Feedback,Unnamed: 13
0,20,Female,Single,Student,No Income,Post Graduate,4,Frequent,12.9766,77.5993,560001,Yes,Positive,Yes
1,24,Female,Single,Student,Below Rs.10000,Graduate,3,Regular,12.9770,77.5773,560009,Yes,Positive,Yes
2,22,Male,Single,Student,Below Rs.10000,Post Graduate,3,Regular,12.9551,77.6593,560017,Yes,Negative,Yes
3,22,Female,Single,Student,No Income,Graduate,6,Frequent,12.9473,77.5616,560019,Yes,Positive,Yes
4,22,Male,Single,Student,Below Rs.10000,Post Graduate,4,Frequent,12.9850,77.5533,560010,Yes,Positive,Yes


In [6]:
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 388 entries, 0 to 387
Data columns (total 14 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   Age                         388 non-null    int64  
 1   Gender                      388 non-null    str    
 2   Marital Status              388 non-null    str    
 3   Occupation                  388 non-null    str    
 4   Monthly Income              388 non-null    str    
 5   Educational Qualifications  388 non-null    str    
 6   Family size                 388 non-null    int64  
 7   Customer Type               388 non-null    str    
 8   latitude                    388 non-null    float64
 9   longitude                   388 non-null    float64
 10  Pin code                    388 non-null    int64  
 11  Output                      388 non-null    str    
 12  Feedback                    388 non-null    str    
 13  Unnamed: 13                 388 non-null    st

In [7]:
# ============================================
# 3. CHECK DATA
# ============================================

print("Rows:", len(df))
print("Columns:", len(df.columns))

print("\nMissing Values:")
print(df.isnull().sum())

print("\nDuplicate Rows:")
print(df.duplicated().sum())


# ============================================
# 4. MAKE A COPY FOR CLEANING
# ============================================

food = df.copy()


# ============================================
# 5. DROP REDUNDANT COLUMN
# ============================================

if "Unnamed: 13" in food.columns:
    food.drop(columns=["Unnamed: 13"], inplace=True)


# ============================================
# 6. HANDLE MISSING VALUES
# ============================================

food["Gender"] = food["Gender"].fillna("Unknown")
food["Marital Status"] = food["Marital Status"].fillna("Unknown")
food["Occupation"] = food["Occupation"].fillna("Unknown")
food["Monthly Income"] = food["Monthly Income"].fillna("Unknown")
food["Educational Qualifications"] = food["Educational Qualifications"].fillna("Unknown")
food["Customer Type"] = food["Customer Type"].fillna("Unknown")
food["Output"] = food["Output"].fillna("Unknown")
food["Feedback"] = food["Feedback"].fillna("Unknown")

food["Age"] = food["Age"].fillna(food["Age"].median())
food["Family size"] = food["Family size"].fillna(food["Family size"].median())
food["latitude"] = food["latitude"].fillna(0)
food["longitude"] = food["longitude"].fillna(0)


# ============================================
# 7. REMOVE DUPLICATES
# ============================================

food.drop_duplicates(inplace=True)


# ============================================
# 8. CLEAN TEXT COLUMNS
# ============================================

for col in food.select_dtypes(include="object").columns:
    food[col] = food[col].astype(str).str.strip()


# ============================================
# 9. CLEAN COLUMN NAMES
# ============================================

food.columns = (
    food.columns
    .str.lower()
    .str.strip()
    .str.replace(" ", "_")
)


# ============================================
# 10. FINAL CHECK
# ============================================

print("Original rows:", len(df))
print("Cleaned rows:", len(food))
print("Columns:", len(food.columns))

print("\nMissing values after cleaning:")
print(food.isnull().sum()[food.isnull().sum() > 0])

print("\nDuplicate rows after cleaning:")
print(food.duplicated().sum())

print("\nData types:")
print(food.dtypes)

food.head()


Rows: 388
Columns: 14

Missing Values:
Age                           0
Gender                        0
Marital Status                0
Occupation                    0
Monthly Income                0
Educational Qualifications    0
Family size                   0
Customer Type                 0
latitude                      0
longitude                     0
Pin code                      0
Output                        0
Feedback                      0
Unnamed: 13                   0
dtype: int64

Duplicate Rows:
103
Original rows: 388
Cleaned rows: 285
Columns: 13

Missing values after cleaning:
Series([], dtype: int64)

Duplicate rows after cleaning:
0

Data types:
age                             int64
gender                            str
marital_status                    str
occupation                        str
monthly_income                    str
educational_qualifications        str
family_size                     int64
customer_type                     str
latitude              

C:\Users\micro\AppData\Local\Temp\ipykernel_15564\569294683.py:60: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in food.select_dtypes(include="object").columns:


,age,gender,marital_status,occupation,monthly_income,educational_qualifications,family_size,customer_type,latitude,longitude,pin_code,output,feedback
0,20,Female,Single,Student,No Income,Post Graduate,4,Frequent,12.9766,77.5993,560001,Yes,Positive
1,24,Female,Single,Student,Below Rs.10000,Graduate,3,Regular,12.9770,77.5773,560009,Yes,Positive
2,22,Male,Single,Student,Below Rs.10000,Post Graduate,3,Regular,12.9551,77.6593,560017,Yes,Negative
3,22,Female,Single,Student,No Income,Graduate,6,Frequent,12.9473,77.5616,560019,Yes,Positive
4,22,Male,Single,Student,Below Rs.10000,Post Graduate,4,Frequent,12.9850,77.5533,560010,Yes,Positive


In [11]:
# ============================================
# 11. CONNECT TO MYSQL
# ============================================

from sqlalchemy import create_engine, text

username = "root"
password = "2005"
host = "localhost"
port = 3306


In [12]:
# ============================================
# 12. CREATE DATABASE
# ============================================

server_engine = create_engine(
    f"mysql+pymysql://{username}:{password}@{host}:{port}"
)

with server_engine.connect() as connection:
    connection.execute(
        text("CREATE DATABASE IF NOT EXISTS food_delivery_analytics")
    )


In [13]:
# ============================================
# 13. CONNECT TO DATABASE
# ============================================

db_engine = create_engine(
    f"mysql+pymysql://{username}:{password}@{host}:{port}/food_delivery_analytics"
)


In [14]:
# ============================================
# 14. LOAD DATA INTO MYSQL
# ============================================

food.to_sql(
    "food_delivery_customers",
    con=db_engine,
    if_exists="replace",
    index=False,
    chunksize=5000
)


285

In [15]:
# ============================================
# 15. VERIFY MYSQL DATA
# ============================================

print(
    pd.read_sql(
        "SELECT COUNT(*) AS total_rows FROM food_delivery_customers",
        db_engine
    )
)

print(
    pd.read_sql(
        "SELECT * FROM food_delivery_customers LIMIT 5",
        db_engine
    )
)


   total_rows
0         285
   age  gender marital_status occupation  monthly_income  \
0   20  Female         Single    Student       No Income   
1   24  Female         Single    Student  Below Rs.10000   
2   22    Male         Single    Student  Below Rs.10000   
3   22  Female         Single    Student       No Income   
4   22    Male         Single    Student  Below Rs.10000   

  educational_qualifications  family_size customer_type  latitude  longitude  \
0              Post Graduate            4      Frequent   12.9766    77.5993   
1                   Graduate            3       Regular   12.9770    77.5773   
2              Post Graduate            3       Regular   12.9551    77.6593   
3                   Graduate            6      Frequent   12.9473    77.5616   
4              Post Graduate            4      Frequent   12.9850    77.5533   

   pin_code output  feedback  
0    560001    Yes  Positive  
1    560009    Yes  Positive  
2    560017    Yes  Negative  
3    5